# [실습1] LangGraph와 Conditional Edge (5개)

## 실습 목표
- LangGraph의 기본 개념 이해
- State와 Node 정의 방법
- **5개 이상의 Conditional Edge 구현**
- 동적 라우팅 패턴 학습

## 0. 환경 설정

In [ ]:
import os
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv()

print(f"✓ 프로젝트 루트: {project_root}")
print(f"✓ OPENAI_API_KEY: {'설정됨' if os.getenv('OPENAI_API_KEY') else '미설정'}")

## 1. LangGraph 기본 개념

### Chain vs Graph
- **Chain**: 순차적(선형) 처리
- **Graph**: 조건부 분기, 병렬 처리 가능

### 주요 요소
1. **State**: 전체 시스템에서 공유되는 데이터
2. **Node**: State를 처리하는 함수
3. **Edge**: Node 간의 연결
4. **Conditional Edge**: 조건에 따라 분기하는 엣지

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, END

# State 정의
class SimpleState(TypedDict):
    query: str
    route: str
    result: str

print("✓ SimpleState 정의 완료")

## 2. Conditional Edge 5개 구현

### Edge 1: 거래 유형 기반 분기
- wire_transfer, crypto, investment, normal → 다른 분석 경로

In [ ]:
# Conditional Edge 1: 거래 유형별 분기
def route_by_transaction_type(state):
    """거래 유형에 따라 분기"""
    query = state['query'].lower()
    
    if 'wire' in query or 'transfer' in query:
        return 'wire_transfer'
    elif 'crypto' in query:
        return 'crypto'
    elif 'invest' in query:
        return 'investment'
    else:
        return 'normal'

# 테스트
test_state1 = {'query': 'wire transfer 거래를 분석해줘', 'route': '', 'result': ''}
print(f"Query: '{test_state1['query']}'")
print(f"Route: {route_by_transaction_type(test_state1)}")

### Edge 2: 거래액 기반 분기
- 고액(>200k) vs 저액(<=200k)

In [ ]:
# Conditional Edge 2: 거래액 기반 분기
def route_by_amount(state):
    """거래액에 따라 분기"""
    # 실제로는 transaction 객체에서 추출
    if any(word in state['query'].lower() for word in ['고액', 'large', '200000', '500000']):
        return 'high_amount'
    else:
        return 'low_amount'

test_state2 = {'query': '$500,000 고액 거래', 'route': '', 'result': ''}
print(f"Query: '{test_state2['query']}'")
print(f"Route: {route_by_amount(test_state2)}")

### Edge 3: 거래 빈도 기반 분기
- 빈번(>10회/24h) vs 정상(<=10회)

In [ ]:
# Conditional Edge 3: 거래 빈도 기반 분기
def route_by_frequency(state):
    """거래 빈도에 따라 분기"""
    if any(word in state['query'].lower() for word in ['빈번', 'frequent', '20회', '30회']):
        return 'frequent'
    else:
        return 'normal_frequency'

test_state3 = {'query': '빈번한 거래 패턴', 'route': '', 'result': ''}
print(f"Query: '{test_state3['query']}'")
print(f"Route: {route_by_frequency(test_state3)}")

### Edge 4: 위험 국가 기반 분기
- 고위험 국가(RU, KP, IR) vs 일반 국가

In [ ]:
# Conditional Edge 4: 위험 국가 기반 분기
def route_by_risk_country(state):
    """위험 국가에 따라 분기"""
    high_risk = ['러시아', 'russia', 'RU', '북한', 'NK', '이란', 'iran']
    
    if any(country in state['query'].upper() for country in high_risk):
        return 'high_risk_country'
    else:
        return 'normal_country'

test_state4 = {'query': '러시아에서 미국으로의 송금', 'route': '', 'result': ''}
print(f"Query: '{test_state4['query']}'")
print(f"Route: {route_by_risk_country(test_state4)}")

### Edge 5: 채널 기반 분기
- 온라인, 오프라인, 암호화폐 채널별 분류

In [ ]:
# Conditional Edge 5: 채널 기반 분기
def route_by_channel(state):
    """거래 채널에 따라 분기"""
    query = state['query'].lower()
    
    if 'crypto' in query or '암호화폐' in state['query']:
        return 'crypto_channel'
    elif 'online' in query or '온라인' in state['query']:
        return 'online_channel'
    else:
        return 'offline_channel'

test_state5 = {'query': '암호화폐 거래소 거래', 'route': '', 'result': ''}
print(f"Query: '{test_state5['query']}'")
print(f"Route: {route_by_channel(test_state5)}")

## 3. 그래프 구성 및 시각화

In [ ]:
# 그래프 흐름도 (ASCII)
graph_flow = """
┌─────────────┐
│  User Query │
└──────┬──────┘
       │
   ┌───▼───┐
   │ init  │
   └───┬───┘
       │
   ┌───▼────────────────────────────────┐
   │ Conditional Edge (5개)             │
   │ 1. Transaction Type               │
   │ 2. Amount                         │
   │ 3. Frequency                      │
   │ 4. Risk Country                   │
   │ 5. Channel                        │
   └───┬────────────────────────────────┘
       │
   ┌───▼─────────────────────┐
   │ Analysis Nodes (5+)     │
   │ - wire_transfer         │
   │ - crypto                │
   │ - high_risk_country     │
   │ - frequent              │
   │ - online_channel        │
   └───┬─────────────────────┘
       │
   ┌───▼────────┐
   │ Response   │
   └────────────┘
"""

print(graph_flow)

## 4. 실제 LangGraph 구현

이 부분은 src/aml_chatbot.py에서 완전히 구현되어 있습니다!

In [ ]:
from src.aml_chatbot import build_chatbot_graph, chat

# 그래프 빌드 확인
graph = build_chatbot_graph()
print("✓ LangGraph 그래프 구축 완료")
print(f"✓ 그래프 타입: {type(graph)}")

## 5. Conditional Edge 동작 테스트

In [ ]:
# 테스트 시나리오
test_queries = [
    ("wire transfer 거래 보안은?", "Edge 1: Transaction Type"),
    ("$500,000 고액 거래 위험도?", "Edge 2: Amount"),
    ("빈번한 거래 패턴이란?", "Edge 3: Frequency"),
    ("러시아 거래 규제는?", "Edge 4: Risk Country"),
    ("암호화폐 거래 안전성?", "Edge 5: Channel"),
]

print("=" * 60)
print("Conditional Edge 동작 테스트")
print("=" * 60)

for query, edge_name in test_queries:
    print(f"\n[{edge_name}]")
    print(f"Q: {query}")
    response = chat(query)
    print(f"A: {response[:100]}...")

## 6. 학습 내용 정리

### LangGraph의 5개 Conditional Edge

| Edge | 분기 기준 | 목적 |
|------|---------|------|
| 1 | 거래 유형 | 거래 종류별 맞춤 분석 |
| 2 | 거래액 | 고액 거래 집중 감시 |
| 3 | 거래 빈도 | Layering 자금세탁 탐지 |
| 4 | 위험 국가 | 규제 대상국 거래 모니터링 |
| 5 | 채널 | 채널별 위험 관리 |

### 핵심 개념
- ✅ State: 전역 데이터 관리
- ✅ Node: 상태 변환 함수
- ✅ Conditional Edge: 동적 라우팅
- ✅ Graph.invoke(): 그래프 실행